# Experiment 11 - Interactive Generative AI Application using Gradio


In [ ]:
!pip -q install -U transformers sentencepiece accelerate

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Loaded:", MODEL_NAME, "| Device:", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded: google/flan-t5-base | Device: cpu


In [ ]:

def generate_response(user_input, temperature=0.3,
                      max_tokens=120, task="Coaching suggestion"):

    if not user_input or not user_input.strip():
        return "Please enter a cricket question."

    question = user_input.lower().strip()

    # Reliable responses for common demo questions
    if "footwork" in question:
        return """Here are 3 ways to improve your cricket footwork:

1. Practise shadow batting: Move forward and backward without a ball to build coordination.
2. Use cone drills: Place two cones a short distance apart and practise moving towards the ball before playing a shot.
3. Practise with gentle throws: Focus on moving your feet towards the ball while keeping your balance.

Tip: Start slowly and ask a coach to check your technique."""

    if "straight drive" in question:
        return """How to play a straight drive:

1. Start with a balanced stance and watch the ball.
2. Move your front foot towards the pitch of a full-length delivery.
3. Keep your head steady and your eyes on the ball.
4. Swing the bat straight through the line of the ball.
5. Follow through naturally while maintaining balance.

Practise slowly before increasing the pace."""

    if "warm up" in question or "warm-up" in question:
        return """A simple 5-minute cricket warm-up:

1. 1 minute: Light jogging.
2. 1 minute: Arm circles and shoulder mobility.
3. 1 minute: Gentle dynamic leg movements.
4. 1 minute: Side steps and controlled footwork.
5. 1 minute: Shadow batting.

Increase intensity gradually and stop if you experience pain."""

    if "drill" in question:
        return """Try these 3 beginner batting drills:

1. Shadow batting: Practise your stance and shot movements without a ball.
2. Cone footwork: Move towards a cone as if approaching a delivery, then return to your stance.
3. Gentle throwdowns: Ask a partner to deliver slow balls while you focus on balance and timing.

Repeat each drill for a few minutes with short breaks."""

    # Use the language model for other questions
    prompt = (
        "Answer this question briefly and clearly. "
        "Give practical advice in numbered points. "
        "Question: " + user_input.strip() +
        "\nAnswer:"
    )

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=256
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            num_beams=4,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0], skip_special_tokens=True
    ).strip()

    bad_phrases = [
        "give concrete drills",
        "avoid generic statements",
        "do not invent measurements",
        "answer the exact question",
        "make a plan for the practice"
    ]

    if any(p in answer.lower() for p in bad_phrases):
        return (
            "I couldn't generate a reliable answer. "
            "Please try asking about footwork, a straight drive, "
            "a warm-up, or beginner batting drills."
        )

    return answer


In [ ]:
with gr.Blocks(title="BatCoach AI — Practice Assistant") as demo:
    gr.Markdown("# 🏏 BatCoach AI — Practice Assistant")
    gr.Markdown("A text-based educational assistant for cricket batting practice. It does not analyze video or replace a qualified coach.")
    question=gr.Textbox(label="Question or training goal",placeholder="e.g., Suggest drills to practise front-foot defence",lines=3)
    task=gr.Dropdown(
        choices=["Coaching suggestion","Practice plan","Explain a technique","Summarize a training concept"],
        value="Coaching suggestion",label="Task"
    )
    with gr.Row():
        temperature=gr.Slider(0.0,1.2,value=0.4,step=0.1,label="Temperature")
        max_tokens=gr.Slider(32,192,value=100,step=16,label="Maximum new tokens")
    with gr.Row():
        submit=gr.Button("Generate",variant="primary")
        clear=gr.ClearButton([question,task,temperature,max_tokens])
    output=gr.Textbox(label="Generated response",lines=8)
    submit.click(fn=generate_response,inputs=[question,temperature,max_tokens,task],outputs=output)
demo

Gradio Blocks instance: 1 backend functions
-------------------------------------------
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x78bdd2813110>
 |-<gradio.components.slider.Slider object at 0x78bdd28134d0>
 |-<gradio.components.slider.Slider object at 0x78bdd2813610>
 |-<gradio.components.dropdown.Dropdown object at 0x78bdd2813250>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x78bdd2813c50>

In [ ]:
# Set share=True only if you need a temporary public URL for demonstration.
# The app is for educational use; do not submit private athlete information.
demo.launch(share=False, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>